# 混合精度与性能边界

## 学习目标

能够解释 autocast 和梯度缩放的职责，并在无 CUDA 时安全跳过。


## 概念模型与执行路径

autocast 根据算子选择较低或较高精度；GradScaler 放大损失，降低 float16 小梯度下溢风险。AMP 主要面向 CUDA，不能假设所有设备具有相同行为。


### 实验 1


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())


### 实验 2


In [ ]:
if torch.cuda.is_available():
    model = torch.nn.Linear(16, 4).cuda()
    inputs = torch.randn(8, 16, device="cuda")
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        outputs = model(inputs)
    print("autocast output dtype:", outputs.dtype)
else:
    print("跳过 CUDA autocast；其他课程仍可在 CPU/MPS 运行。")


### 实验 3


In [ ]:
if torch.cuda.is_available():
    optimizer = torch.optim.AdamW(model.parameters())
    scaler = torch.amp.GradScaler("cuda")
    targets = torch.randn_like(outputs)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        loss = torch.nn.functional.mse_loss(model(inputs), targets)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    print("scaled step complete:", loss.item())


### 实验 4


In [ ]:
# python 07-deep-learning/pytorch/examples/mixed_precision.py --device auto --quick


## 底层机制

模型参数通常保留 float32，autocast 控制算子输入与输出精度。梯度缩放只解决下溢，不解决学习率错误或数值不稳定模型。性能收益取决于硬件、模型规模和数据管线。


## 检查点

autocast 和 GradScaler 分别解决什么问题？为什么 CPU 上跳过 CUDA AMP 不应视为测试失败？


## 试一试

在 CUDA 上记录开启/关闭 AMP 的迭代时间和峰值显存；先预热再测量，并同步 CUDA。


## 常见错误与调试

手动把全部参数转成 half、在 CPU 强行使用 CUDA scaler、计时前未同步、认为 AMP 一定提高小模型速度。
